<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Finance (2nd ed.)
**Mastering Data-Driven Finance**
&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH
<img src="http://hilpisch.com/images/py4fi_2nd_shadow.png" width="300px" align="left">

# Automated Trading

## Risk Disclaimer

<font size="-1">
Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits. Leverage can work against you. Due to the certain restrictions imposed by the local law and regulation, German resident retail client(s) could sustain a total loss of deposited funds but are not subject to subsequent payment obligations beyond the deposited funds. Be aware and fully understand all risks associated with the market and trading. Prior to trading any products, carefully consider your financial situation and experience level. Any opinions, news, research, analyses, prices, or other information is provided as general market commentary, and does not constitute investment advice. FXCM & TPQ will not accept liability for any loss or damage, including without limitation to, any loss of profit, which may arise directly or indirectly from use of or reliance on such information.
</font>

## Author Disclaimer

The author is neither an employee, agent nor representative of FXCM and is therefore acting independently. The opinions given are their own, constitute general market commentary, and do not constitute the opinion or advice of FXCM or any form of personal or investment advice. FXCM assumes no responsibility for any loss or damage, including but not limited to, any loss or gain arising out of the direct or indirect use of this or any other content. Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits.

In [ ]:
import math
import time
import numpy as np
import pandas as pd
import datetime as dt
import cufflinks as cf
from pylab import mpl, plt

In [ ]:
np.random.seed(1000)
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'

## Capital Management

### Kelly Criterion in a Binomial Setting

In [ ]:
p = 0.55

In [ ]:
f = p - (1 - p)

In [ ]:
f

In [ ]:
I = 50

In [ ]:
n = 100

In [ ]:
def run_simulation(f):
    c = np.zeros((n, I))
    c[0] = 100
    for i in range(I):
        for t in range(1, n):
            o = np.random.binomial(1, p)
            if o > 0:
                c[t, i] = (1 + f) * c[t - 1, i]
            else:
                c[t, i] = (1 - f) * c[t - 1, i]
    return c

In [ ]:
c_1 = run_simulation(f)

In [ ]:
c_1.round(2)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(c_1, 'b', lw=0.5)
plt.plot(c_1.mean(axis=1), 'r', lw=2.5);
# plt.savefig('../../images/ch16/auto_plot_01.png');

In [ ]:
c_2 = run_simulation(0.05)

In [ ]:
c_3 = run_simulation(0.25)

In [ ]:
c_4 = run_simulation(0.5)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(c_1.mean(axis=1), 'r', label='$f^*=0.1$')
plt.plot(c_2.mean(axis=1), 'b', label='$f=0.05$')
plt.plot(c_3.mean(axis=1), 'y', label='$f=0.25$')
plt.plot(c_4.mean(axis=1), 'm', label='$f=0.5$')
plt.legend(loc=0);

### Kelly Criterion for Stocks and Indices

In [ ]:
raw = pd.read_csv('../../source/tr_eikon_eod_data.csv',
                  index_col=0, parse_dates=True)

In [ ]:
symbol = '.SPX'

In [ ]:
data = pd.DataFrame(raw[symbol])

In [ ]:
data['returns'] = np.log(data / data.shift(1))

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.tail()

In [ ]:
mu = data.returns.mean() * 252

In [ ]:
mu

In [ ]:
sigma = data.returns.std() * 252 ** 0.5

In [ ]:
sigma

In [ ]:
r = 0.0

In [ ]:
f = (mu - r) / sigma ** 2

In [ ]:
f

In [ ]:
equs = []

In [ ]:
def kelly_strategy(f):
    global equs
    equ = 'equity_{:.2f}'.format(f)
    equs.append(equ)
    cap = 'capital_{:.2f}'.format(f)
    data[equ] = 1.
    data[cap] = data[equ] * f
    for i, t in enumerate(data.index[1:]):
        t_1 = data.index[i]
        data.loc[t, cap] = data[cap].loc[t_1] * \
                            math.exp(data['returns'].loc[t])
        data.loc[t, equ] = data[cap].loc[t] - \
                            data[cap].loc[t_1] + \
                            data[equ].loc[t_1]
        data.loc[t, cap] = data[equ].loc[t] * f

In [ ]:
kelly_strategy(f * 0.5)

In [ ]:
kelly_strategy(f * 0.66)

In [ ]:
kelly_strategy(f)

In [ ]:
print(data[equs].tail())

In [ ]:
ax = data['returns'].cumsum().apply(np.exp).plot(legend=True, figsize=(10, 6))
data[equs].plot(ax=ax, legend=True);

## ML-Based Trading Strategy

<b style="color: red; font-size: 16px;">FXCM has stopped the original API support.<br>Therefore the rest of the code in this notebook does unfortunately not work anymore.</b>

### Vectorized Backtesting

In [ ]:
import fxcmpy

In [ ]:
fxcmpy.__version__

In [ ]:
%time api = fxcmpy.fxcmpy(config_file='../../cfg/fxcm.cfg')

In [ ]:
data = api.get_candles('EUR/USD', period='m5',
                        start='2018-06-01 00:00:00',
                        stop='2018-06-30 00:00:00')

In [ ]:
data.iloc[-5:, 4:]

In [ ]:
data.info()

In [ ]:
spread = (data['askclose'] - data['bidclose']).mean()
spread

In [ ]:
data['midclose'] = (data['askclose'] + data['bidclose']) / 2

In [ ]:
ptc = spread / data['midclose'].mean()
ptc

In [ ]:
data['midclose'].plot(figsize=(10, 6), legend=True);
# plt.savefig('../../images/ch16/auto_plot_04.png');

In [ ]:
data['returns'] = np.log(data['midclose'] / data['midclose'].shift(1))

In [ ]:
data.dropna(inplace=True)

In [ ]:
lags = 5

In [ ]:
cols = []
for lag in range(1, lags + 1):
    col = 'lag_{}'.format(lag)
    data[col] = data['returns'].shift(lag)
    cols.append(col)

In [ ]:
data.dropna(inplace=True)

In [ ]:
data[cols] = np.where(data[cols] > 0, 1, 0)

In [ ]:
data['direction'] = np.where(data['returns'] > 0, 1, -1)

In [ ]:
data[cols + ['direction']].head()

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [ ]:
model = SVC(C=1, kernel='linear')

In [ ]:
split = int(len(data) * 0.80)

In [ ]:
train = data.iloc[:split].copy()

In [ ]:
model.fit(train[cols], train['direction'])

In [ ]:
accuracy_score(train['direction'], model.predict(train[cols]))

In [ ]:
test = data.iloc[split:].copy()

In [ ]:
test['position'] = model.predict(test[cols])

In [ ]:
accuracy_score(test['direction'], test['position'])

In [ ]:
test['strategy'] = test['position'] * test['returns']

In [ ]:
sum(test['position'].diff() != 0)

In [ ]:
test['strategy_tc'] = np.where(test['position'].diff() != 0,
                               test['strategy'] - ptc,
                               test['strategy'])

In [ ]:
test[['returns', 'strategy', 'strategy_tc']].sum(
        ).apply(np.exp)

In [ ]:
test[['returns', 'strategy', 'strategy_tc']].cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));
# plt.savefig('../../images/ch16/auto_plot_05.png');

In [ ]:
mean = test[['returns', 'strategy_tc']].mean() * len(data) * 12
mean

In [ ]:
var = test[['returns', 'strategy_tc']].var() * len(data) * 12
var

In [ ]:
vol = var ** 0.5
vol

In [ ]:
mean / var

In [ ]:
mean / var * 0.5

In [ ]:
to_plot = ['returns', 'strategy_tc']

In [ ]:
for lev in [10, 20, 30, 40, 50]:
    label = 'lstrategy_tc_%d' % lev
    test[label] = test['strategy_tc'] * lev
    to_plot.append(label)

In [ ]:
test[to_plot].cumsum().apply(np.exp).plot(figsize=(10, 6));

### Risk Analysis

In [ ]:
equity = 3333

In [ ]:
risk = pd.DataFrame(test['lstrategy_tc_30'])

In [ ]:
risk['equity'] = risk['lstrategy_tc_30'].cumsum().apply(np.exp) * equity

In [ ]:
risk['cummax'] = risk['equity'].cummax()

In [ ]:
risk['drawdown'] = risk['cummax'] - risk['equity']

In [ ]:
risk['drawdown'].max()

In [ ]:
t_max = risk['drawdown'].idxmax()
t_max

In [ ]:
temp = risk['drawdown'][risk['drawdown'] == 0]

In [ ]:
periods = (temp.index[1:].to_pydatetime() -
           temp.index[:-1].to_pydatetime())

In [ ]:
periods[20:30]

In [ ]:
t_per = periods.max()

In [ ]:
t_per

In [ ]:
t_per.seconds / 60 / 60

In [ ]:
risk[['equity', 'cummax']].plot(figsize=(10, 6))
plt.axvline(t_max, c='r', alpha=0.5);

In [ ]:
import scipy.stats as scs

In [ ]:
percs = [0.01, 0.1, 1., 2.5, 5.0, 10.0]

In [ ]:
risk['returns'] = np.log(risk['equity'] /
                         risk['equity'].shift(1))

In [ ]:
VaR = scs.scoreatpercentile(equity * risk['returns'], percs)

In [ ]:
def print_var():
    print('%16s %16s' % ('Confidence Level', 'Value-at-Risk'))
    print(33 * '-')
    for pair in zip(percs, VaR):
        print('%16.2f %16.3f' % (100 - pair[0], -pair[1]))

In [ ]:
print_var()

In [ ]:
hourly = risk.resample('1H', label='right').last()

In [ ]:
hourly['returns'] = np.log(hourly['equity'] /
                         hourly['equity'].shift(1))

In [ ]:
VaR = scs.scoreatpercentile(equity * hourly['returns'], percs)

In [ ]:
print_var()

### Persisting the Model Object

In [ ]:
import pickle

In [ ]:
pickle.dump(model, open('algorithm.pkl', 'wb'))

## Online Algorithm

In [ ]:
algorithm = pickle.load(open('algorithm.pkl', 'rb'))

In [ ]:
algorithm

In [ ]:
sel = ['tradeId', 'amountK', 'currency',
       'grossPL', 'isBuy']

In [ ]:
def print_positions(pos):
    print('\n\n' + 50 * '=')
    print('Going {}.\n'.format(pos))
    time.sleep(1.5)
    print(api.get_open_positions()[sel])
    print(50 * '=' + '\n\n')

In [ ]:
symbol = 'EUR/USD'
bar = '15s'
amount = 100
position = 0
min_bars = lags + 1
df = pd.DataFrame()

In [ ]:
def automated_strategy(data, dataframe):
    global min_bars, position, df
    ldf = len(dataframe)
    df = dataframe.resample(bar, label='right').last().ffill()
    if ldf % 20 == 0:
        print('%3d' % len(dataframe), end=',')
    if len(df) > min_bars:
        min_bars = len(df)
        df['Mid'] = df[['Bid', 'Ask']].mean(axis=1)
        df['Returns'] = np.log(df['Mid'] / df['Mid'].shift(1))
        df['Direction'] = np.where(df['Returns'] > 0, 1, -1)
        features = df['Direction'].iloc[-(lags + 1):-1]
        features = features.values.reshape(1, -1)
        signal = algorithm.predict(features)[0]
        if position in [0, -1] and signal == 1:
            api.create_market_buy_order(
                symbol, amount - position * amount)
            position = 1
            print_positions('LONG')
        elif position in [0, 1] and signal == -1:
            api.create_market_sell_order(
                symbol, amount + position * amount)
            position = -1
            print_positions('SHORT')
    if len(dataframe) > 350:
        api.unsubscribe_market_data('EUR/USD')
        api.close_all()

In [ ]:
# api.subscribe_market_data(symbol, (automated_strategy,))

In [ ]:
# api.unsubscribe_market_data(symbol)

In [ ]:
# api.close_all()

## Logging and Monitoring

In [ ]:
!cat automated_strategy.py

In [ ]:
!cat strategy_monitoring.py

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>
<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>